# Mercari Price Regression — runnable notebook

This notebook is a cell-by-cell version of the provided scripts:

- `dataset.py` (dataset + text building)
- `model.py` (transformer + regression head)
- `evaluate.py` (RMSE/MAE and prediction helpers)
- `train.py` (training loop and config)

It’s designed so you can run sequentially, tweak config, and re-train/evaluate/predict.


## 0) Install dependencies (optional)

If you're running locally and your environment doesn't already have the required packages, run the next cell.


In [ ]:
# If needed (e.g., in a fresh venv / Colab), uncomment:
# !pip install -r requirements.txt


## 1) Imports + basic setup


In [ ]:
import os
import json
import math
import random
from dataclasses import dataclass, asdict
from typing import Dict, Optional, Any

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


## 2) Reproducibility helpers + device


In [ ]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## 3) Dataset utilities (from `dataset.py`)


In [ ]:
def build_text(name: str, desc: str) -> str:
    name = "" if pd.isna(name) else str(name)
    desc = "" if pd.isna(desc) else str(desc)
    # keep a separator token-ish marker; the tokenizer will handle it as normal text
    return f"{name} [SEP] {desc}".strip()


class MercariTextDataset(Dataset):
    """
    Uses only:
      - name
      - item_description
      - price (optional; present for train/val)
    Produces tokenized tensors + optional labels (log1p(price)).
    """
    def __init__(
        self,
        df: pd.DataFrame,
        tokenizer,
        max_length: int = 128,
        has_labels: bool = True,
        price_col: str = "price",
    ):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.has_labels = has_labels
        self.price_col = price_col

        # Prebuild texts once (faster + deterministic)
        self.texts = [
            build_text(n, d)
            for n, d in zip(self.df["name"], self.df["item_description"])
        ]

        if self.has_labels:
            if self.price_col not in self.df.columns:
                raise ValueError(f"has_labels=True but '{self.price_col}' not in df columns")
            # log1p is standard for price regression
            self.labels = np.log1p(self.df[self.price_col].astype(float).values)
        else:
            self.labels = None

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        text = self.texts[idx]

        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        item = {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }

        if self.has_labels:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float32)

        return item


## 4) Model (from `model.py`)


In [ ]:
from dataclasses import dataclass

@dataclass
class ModelOutput:
    preds: torch.Tensor
    loss: Optional[torch.Tensor]


class PriceRegressor(nn.Module):
    """
    Fine-tunes transformer + regression head.
    Input: tokenized text (name + description)
    Output: log1p(price)
    """
    def __init__(
        self,
        encoder_name: str,
        dropout: float = 0.1,
        hidden_dim: int = 256,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(encoder_name)
        h = self.encoder.config.hidden_size

        self.head = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(h, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1),
        )
        self.loss_fn = nn.MSELoss()

    def forward(self, input_ids, attention_mask, labels=None) -> ModelOutput:
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]          # [B, H]
        pred = self.head(cls).squeeze(-1)             # [B]

        loss = None
        if labels is not None:
            loss = self.loss_fn(pred, labels)

        return ModelOutput(preds=pred, loss=loss)


## 5) Evaluation helpers (from `evaluate.py`)


In [ ]:
@torch.no_grad()
def evaluate_loader(model, loader, device: torch.device) -> Dict[str, float]:
    """
    Requires labels to exist in loader batches.
    Computes RMSE/MAE on original price scale by expm1.
    """
    model.eval()

    preds_log, labels_log = [], []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=None)

        preds_log.append(out.preds.detach().cpu().numpy())
        labels_log.append(labels.detach().cpu().numpy())

    preds_log = np.concatenate(preds_log)
    labels_log = np.concatenate(labels_log)

    preds = np.expm1(preds_log)
    labels = np.expm1(labels_log)

    rmse = float(np.sqrt(np.mean((preds - labels) ** 2)))
    mae = float(np.mean(np.abs(preds - labels)))
    return {"rmse": rmse, "mae": mae}


@torch.no_grad()
def predict_loader(model, loader, device: torch.device) -> np.ndarray:
    """
    Works even if loader has no labels (e.g., your test file).
    Returns predictions on original price scale.
    """
    model.eval()
    preds_log = []

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, labels=None)
        preds_log.append(out.preds.detach().cpu().numpy())

    preds_log = np.concatenate(preds_log)
    return np.expm1(preds_log)


def delta_stats(pred_prof: np.ndarray, pred_orig: np.ndarray) -> Dict[str, float]:
    delta = pred_prof - pred_orig
    return {
        "delta_mean": float(delta.mean()),
        "delta_median": float(np.median(delta)),
        "delta_std": float(delta.std()),
        "pct_positive": float((delta > 0).mean()),
        "pct_negative": float((delta < 0).mean()),
    }


## 6) Config (from `train.py`)


In [ ]:
@dataclass
class TrainConfig:
    encoder_name: str = "distilbert-base-uncased"
    max_length: int = 128

    lr: float = 2e-5
    weight_decay: float = 0.01
    batch_size: int = 16
    num_epochs: int = 3
    warmup_ratio: float = 0.06
    grad_clip: float = 1.0

    num_workers: int = 2
    seed: int = 42
    out_dir: str = "checkpoints/price_model"

cfg = TrainConfig()
cfg


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 7) Load CSVs + minimal cleaning (matches `train.py`)


In [ ]:
# Paths (these sample CSVs are in the same folder as the provided scripts)
BASE_PATH = "/content/drive/MyDrive/Lokaverkefni - Gön"

TRAIN_CSV = "train_sample.csv"
VAL_CSV   = "validation_sample.csv"
TEST_CSV  = "test_sample.csv"

TRAIN_CSV = os.path.join(BASE_PATH, "train_sample.csv")
VAL_CSV   = os.path.join(BASE_PATH, "validation_sample.csv")
TEST_CSV  = os.path.join(BASE_PATH, "test_sample.csv")

print("raw shapes:", train_df.shape, val_df.shape, test_df.shape)
train_df.head()


In [ ]:
# --- minimal cleaning (same as train.py) ---

cols = ["train_id", "name", "item_description", "price"]
train_df = train_df[cols]
val_df   = val_df[cols]

# Remove non-positive prices (recommended)
train_df = train_df[train_df["price"] > 0].reset_index(drop=True)
val_df   = val_df[val_df["price"] > 0].reset_index(drop=True)

# Replace placeholder descriptions
train_df["item_description"] = train_df["item_description"].replace("No description yet", "")
val_df["item_description"]   = val_df["item_description"].replace("No description yet", "")

print("clean shapes:", train_df.shape, val_df.shape)
train_df.sample(3, random_state=0)


## 8) Tokenizer + Datasets + DataLoaders


In [ ]:
set_seed(cfg.seed)

tokenizer = AutoTokenizer.from_pretrained(cfg.encoder_name)

train_ds = MercariTextDataset(train_df, tokenizer, max_length=cfg.max_length, has_labels=True)
val_ds   = MercariTextDataset(val_df, tokenizer, max_length=cfg.max_length, has_labels=True)

# test may or may not have labels; we treat it as unlabeled
# expects columns: name, item_description (plus any ids)
if "item_description" in test_df.columns:
    test_df["item_description"] = test_df["item_description"].replace("No description yet", "")
test_ds = MercariTextDataset(test_df, tokenizer, max_length=cfg.max_length, has_labels=False)

train_loader = DataLoader(
    train_ds, batch_size=cfg.batch_size, shuffle=True,
    num_workers=cfg.num_workers, pin_memory=(device.type == "cuda")
)
val_loader = DataLoader(
    val_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=(device.type == "cuda")
)
test_loader = DataLoader(
    test_ds, batch_size=cfg.batch_size, shuffle=False,
    num_workers=cfg.num_workers, pin_memory=(device.type == "cuda")
)

len(train_loader), len(val_loader), len(test_loader)


## 9) Model + optimizer + scheduler


In [ ]:
model = PriceRegressor(cfg.encoder_name).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

total_steps = cfg.num_epochs * len(train_loader)
warmup_steps = int(cfg.warmup_ratio * total_steps)
scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))

total_steps, warmup_steps


## 10) Training loop (matches `train.py`) — saves best checkpoint


In [ ]:
os.makedirs(cfg.out_dir, exist_ok=True)

with open(os.path.join(cfg.out_dir, "train_config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

best_val_rmse = float("inf")
history = []
history_path = os.path.join(cfg.out_dir, "history.csv")

for epoch in range(1, cfg.num_epochs + 1):
    model.train()
    running = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{cfg.num_epochs}", leave=False)
    for batch_idx, batch in enumerate(pbar, start=1):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=(device.type == "cuda")):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running += float(loss.item())
        avg_so_far = running / batch_idx
        pbar.set_postfix(train_mse=f"{avg_so_far:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    avg_loss = running / max(1, len(train_loader))
    val_metrics = evaluate_loader(model, val_loader, device=device)
    val_rmse = val_metrics["rmse"]

    current_lr = optimizer.param_groups[0]["lr"]
    history.append({
        "epoch": epoch,
        "train_mse": avg_loss,
        "val_rmse": val_metrics["rmse"],
        "val_mae": val_metrics["mae"],
        "lr": current_lr,
    })
    pd.DataFrame(history).to_csv(history_path, index=False)

    print(
        f"Epoch {epoch}/{cfg.num_epochs} | "
        f"train_mse={avg_loss:.4f} | val_rmse={val_rmse:.4f} | val_mae={val_metrics['mae']:.4f}"
    )

    if val_rmse < best_val_rmse:
        best_val_rmse = val_rmse
        ckpt_path = os.path.join(cfg.out_dir, "best.pt")
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "encoder_name": cfg.encoder_name,
                "best_val_rmse": best_val_rmse,
                "epoch": epoch,
            },
            ckpt_path,
        )
        print(f"  ✅ saved best -> {ckpt_path}")

print(f"Done. Best val RMSE: {best_val_rmse:.4f}")


## 11) Load best checkpoint (optional) + re-evaluate


In [ ]:
ckpt_path = os.path.join(cfg.out_dir, "best.pt")
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print("Loaded:", ckpt_path, "| best_val_rmse:", ckpt.get("best_val_rmse"), "| epoch:", ckpt.get("epoch"))
else:
    print("No checkpoint found at", ckpt_path)

evaluate_loader(model, val_loader, device=device)


## 12) Predict on test set + save submission-style CSV


In [ ]:
pred_prices = predict_loader(model, test_loader, device=device)

out_pred_path = os.path.join(cfg.out_dir, "test_predictions.csv")

# Try to preserve an id column if present; else just output row index
id_col = None
for c in ["test_id", "train_id", "id"]:
    if c in test_df.columns:
        id_col = c
        break

out_df = pd.DataFrame({
    (id_col if id_col else "row_id"): (test_df[id_col].values if id_col else np.arange(len(test_df))),
    "price_pred": pred_prices
})

out_df.to_csv(out_pred_path, index=False)
out_df.head(), out_pred_path


## 13) Quick sanity checks / explorations


In [ ]:
# Basic distribution check
out_df["price_pred"].describe()
